# 1.0 Exploración inicial de los viajes de MiBici

Primer contacto con los viajes de 2025. La lectura, la detección de codificación y los umbrales vienen de los módulos de `mibici_mlops`; aquí solo se explora. Lo que se descubrió en esta exploración quedó como reglas en `cleaning.py` y como banderas en la versión oficial.

In [1]:
import pandas as pd
from loguru import logger

from mibici_mlops.config import (
    FALSE_START_MAX_MIN, LONG_TRIP_MIN, RAW_DATA_DIR, SHORT_TRIP_MAX_MIN, STATIONS_FILE,
    TRIPS_PATTERN,
)
from mibici_mlops.dataset import detect_encoding, load_raw_trips, load_stations

logger.disable('mibici_mlops')

2026-11-07 12:35:55.795 | INFO     | mibici_mlops.config:<module>:12 - PROJ_ROOT path is: /home/fgrodriguez/mibici-mlops


/home/fgrodriguez/mibici-mlops/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Codificación

Al abrir los archivos con UTF-8, algunos meses fallan.

In [2]:
archivos = sorted(RAW_DATA_DIR.glob(TRIPS_PATTERN))
pd.Series({f.stem[-7:]: detect_encoding(f) for f in archivos}).value_counts()

utf-8      9
latin-1    3
Name: count, dtype: int64

In [3]:
df, metadatos = load_raw_trips()
print(df.shape)
df.head()

(4532032, 8)


,viaje_id,usuario_id,genero,anio_nacimiento,inicio,fin,origen_id,destino_id
0,37162342,601273,M,1982,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395
1,37162343,1571524,M,2003,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260
2,37162344,1435200,F,1984,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260
3,37162345,126327,M,1988,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11
4,37162346,416121,M,1994,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8


## Valores nulos y texto 'NULL'

El portal escribe los nulos como texto.

In [4]:
pd.DataFrame({'NULL': (df == 'NULL').sum(), 'NA': (df == 'NA').sum(), 'vacío': (df == '').sum()})

,NULL,NA,vacío
viaje_id,0,0,0
usuario_id,0,0,0
genero,40789,0,0
anio_nacimiento,0,1105,0
inicio,0,0,0
fin,0,0,0
origen_id,0,0,0
destino_id,0,0,0


## Año de nacimiento

In [5]:
anio = pd.to_numeric(df['anio_nacimiento'], errors='coerce')
print('no numéricos:', anio.isna().sum())
print('antes de 1930:', (anio < 1930).sum(), ' después de 2010:', (anio > 2010).sum())
anio.value_counts().sort_index().loc[1966:1974]

no numéricos: 1105
antes de 1930: 143  después de 2010: 398


anio_nacimiento
1966.0    20307
1967.0    30247
1968.0    32934
1969.0    36766
1970.0    91466
1971.0    36663
1972.0    36806
1973.0    34527
1974.0    41457
Name: count, dtype: int64

El año 1970 tiene más del doble de viajes que sus vecinos. Al contar usuarios distintos pasa lo mismo, lo que sugiere un valor por omisión en el registro.

In [6]:
u = df.drop_duplicates('usuario_id')
pd.to_numeric(u['anio_nacimiento'], errors='coerce').value_counts().sort_index().loc[1968:1972]

anio_nacimiento
1968.0    138
1969.0    165
1970.0    721
1971.0    171
1972.0    202
Name: count, dtype: int64

## Duración de los viajes

In [7]:
dur = (pd.to_datetime(df['fin']) - pd.to_datetime(df['inicio'])).dt.total_seconds() / 60
dur.describe(percentiles=[.001, .01, .05, .5, .95, .99, .999]).round(2)

count    4532032.00
mean          11.15
std           10.32
min            0.00
0.1%           0.03
1%             0.28
5%             2.47
50%            9.62
95%           24.27
99%           32.75
99.9%         56.43
max        11500.77
dtype: float64

In [8]:
mismo = df['origen_id'] == df['destino_id']
tramos = [(0, SHORT_TRIP_MAX_MIN), (SHORT_TRIP_MAX_MIN, FALSE_START_MAX_MIN),
          (FALSE_START_MAX_MIN, 5), (5, 30), (30, LONG_TRIP_MIN), (LONG_TRIP_MIN, 1e9)]
pd.DataFrame([{'tramo_min': f'({a:g}, {b:g}]', 'misma_estacion': ((dur > a) & (dur <= b) & mismo).sum(), 'distinta_estacion': ((dur > a) & (dur <= b) & ~mismo).sum()} for a, b in tramos])

,tramo_min,misma_estacion,distinta_estacion
0,"(0, 1]",109121,5881
1,"(1, 2]",14173,40728
2,"(2, 5]",13236,665441
3,"(5, 30]",110246,3501292
4,"(30, 180]",7309,61371
5,"(180, 1e+09]",90,364


Los viajes de menos de dos minutos que regresan a la misma estación son, en su mayoría, bicicletas que se sacan y se devuelven de inmediato. Con distinta estación, menos de un minuto no alcanza para recorrer la distancia entre estaciones.

In [9]:
q1, q3 = dur.quantile([.25, .75])
limite = q3 + 1.5 * (q3 - q1)
print('límite superior por IQR:', round(limite, 1), 'min')
print('viajes sobre ese límite:', (dur > limite).sum())
print(f'viajes de más de {LONG_TRIP_MIN:g} min:', (dur > LONG_TRIP_MIN).sum())

límite superior por IQR: 28.5 min
viajes sobre ese límite: 91809
viajes de más de 180 min: 454


El criterio IQR marcaría como atípicos cerca de 92 mil viajes de más de 28.5 minutos, que son plausibles en bicicleta; por eso se usa un umbral de dominio de 180 minutos.

## Estaciones

In [10]:
st = load_stations()
ids = set(st['estacion_id'].astype(str))
print('estaciones en catálogo:', len(st), ' usadas:', df['origen_id'].nunique())
print('origen fuera del catálogo:', (~df['origen_id'].isin(ids)).sum())
print('capacidad 0:', (st['capacidad'] == 0).sum())

estaciones en catálogo: 484  usadas: 371


origen fuera del catálogo: 0
capacidad 0: 4
